# 08 消融实验与综合评价

## 本课学习目标

- A. 量化金融主线：消融实验（Ablation Study）
零基础解释：一次只加入或去掉一个模块，观察它是否真的带来改进。
- B. 大语言模型主线：检索增强生成（Retrieval-Augmented Generation，RAG）和 TF-IDF
零基础解释：本课只演示先检索再使用资料的思想，不实现真实在线 RAG。
- C. 两条线如何连接：把市场数据和新闻文本转成可检查的表格信号。
- D. 可运行实验：比较四组策略，输出净值曲线、指标表、混淆矩阵和检索结果。
- E. 结果解释：观察表格、图表和结构化输出。
- F. 常见错误：把回测收益当成未来收益、把 Mock 当成真实模型。
- G. 课后练习：修改一个参数并重新运行。
- H. 本课术语表：见本课各小节。

## 本课最终输出

一个离线实验输出，不联网、不调用真实模型、不产生真实订单。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "learning").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "learning" / "data"


## 可运行实验

下面代码只读取 `learning/data` 下的合成数据。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from learning.src.market_data import load_price_data
from learning.src.momentum_factor import compute_momentum, rank_momentum
from learning.src.sentiment_factor import classify_news, daily_sentiment_factor
from learning.src.time_alignment import monthly_signal_schedule
from learning.src.mini_backtest import run_backtest
from learning.src.financial_metrics import simple_returns, cumulative_return, maximum_drawdown, sharpe_ratio
from learning.src.retrieval_demo import retrieve_notes
prices = load_price_data(DATA / "sample_prices.csv")
news_raw = pd.read_csv(DATA / "sample_news.csv")
news = classify_news(news_raw)
schedule = monthly_signal_schedule(prices).head(8)
mom = compute_momentum(prices, 20)
def make_targets(mode):
    rows = []
    for _, s in schedule.iterrows():
        base = rank_momentum(mom, s["signal_date"], 20)
        sent = daily_sentiment_factor(news, s["signal_timestamp"]).groupby("ticker")["sentiment_score"].mean()
        base["sentiment_score"] = base["ticker"].map(sent).fillna(0)
        if mode == "buy_hold":
            picks = ["AAA", "BBB"]
        elif mode == "momentum":
            picks = base.nlargest(2, "momentum_rank_score")["ticker"].tolist()
        elif mode == "sentiment":
            picks = base.nlargest(2, "sentiment_score")["ticker"].tolist()
        else:
            base["combined"] = 0.7 * base["momentum_rank_score"] + 0.3 * ((base["sentiment_score"] + 1) / 2)
            picks = base.nlargest(2, "combined")["ticker"].tolist()
        for ticker in picks:
            rows.append({"execution_date": s["execution_date"], "ticker": ticker, "weight": 0.5})
    return pd.DataFrame(rows)
curves = {}
rows = []
for mode in ["buy_hold", "momentum", "sentiment", "combined"]:
    result = run_backtest(prices, make_targets(mode))
    eq = result.equity_curve.set_index("date")["equity"]
    curves[mode] = eq / eq.iloc[0]
    rets = simple_returns(eq)
    rows.append({"mode": mode, "cumulative": cumulative_return(rets), "max_drawdown": maximum_drawdown(eq), "sharpe": sharpe_ratio(rets)})
pd.DataFrame(curves).plot(figsize=(9,4), title="Ablation equity curves")
plt.show()
display(pd.DataFrame(rows))
display(pd.DataFrame(confusion_matrix(news_raw["expected_label"], news["label"], labels=["positive","neutral","negative"]), index=["expected_positive","expected_neutral","expected_negative"], columns=["pred_positive","pred_neutral","pred_negative"]))
display(retrieve_notes(DATA / "sample_company_notes.csv", "model risk and teaching data", 3))
print("结果仅用于教学，不代表未来收益。")

## 结尾总结

你现在应该理解：量化数据和文本模型输出都必须被结构化、校验并按时间对齐。

哪些结果不能解释为策略一定赚钱：任何图表和收益数字都只是合成数据上的教学结果。

本课使用了哪些英文专业词：Large Language Model, Prompt, Structured Output, Backtesting, Factor, Return, Risk。

下一课与本课有什么关系：下一课会在本课结果上继续增加一个新量化概念和一个新 LLM 概念。